# Figures

Consumes cached artefacts from `data/cache/`; one cell per figure group.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent if Path.cwd().name == 'make_figures' else Path.cwd().resolve()))

import importlib

from utils import config
from utils.config import load_colors
from utils import cb_data, core_data, ol_data, ol_rf, polarity, vic
from utils import plotting as plot
plot = importlib.reload(plot)

plot.set_html_output(False)


## 1. Inventory


In [ ]:
inv = core_data.get_inventory_meta()
colored_region, colored_main_groups, colored_sign, _ = load_colors()
colors = {
    'region': colored_region['color'].to_dict(),
    'main_groups': colored_main_groups['color'].to_dict(),
    'sign': colored_sign['color'].to_dict(),
}
plot.plot_inventory_full_brain(inv, colors)
plot.plot_inventory_right_ol(inv, colors)
plot.plot_inventory_by_main_group(inv, colors)
print('wrote figures to', config.FIG_DIR / 'inventory')


## 2. Layers


In [ ]:
# Preprocessing (cached, no recompute if Main 1 ran)
ol_meta     = core_data.get_ol_meta()
ol_flow     = core_data.get_ol_flow_type()
meta        = core_data.get_meta()
flow_df     = core_data.get_flow().merge(meta[['idx','main_groups','sign']], on='idx', how='right')
flow_ol     = core_data.get_flow_ol()
conn_ol     = ol_data.get_ol_connectivity()
dir_ol      = ol_data.get_ol_directedness()
type_dir_ol = ol_data.get_ol_type_directedness()
layers      = ol_data.get_ol_layers()
roi_adj     = ol_data.get_ol_roi_adjacency()
sectors     = core_data.get_sector_map()

colored_region, colored_main_groups, colored_sign, _ = config.load_colors()

plot.plot_coords(ol_meta)
plot.plot_flow_sector_division(sectors)
plot.plot_flow_by_main_groups(ol_flow, conn_ol, type_dir_ol, flow_ol, colored_main_groups)
plot.plot_flow_by_sign(ol_flow, conn_ol, type_dir_ol, flow_ol, colored_sign)
plot.plot_flow_type_scatter_and_triangle(ol_flow, type_dir_ol, colored_main_groups, colored_sign)
plot.plot_flow_type_boxes(flow_df, dir_ol, colored_main_groups)
plot.plot_flow_numerous_type_boxes(flow_ol, dir_ol, colored_main_groups)
plot.plot_flow_connectivity_heatmap(conn_ol, colored_main_groups, colored_sign)
plot.plot_flow_threshold_choice(conn_ol)

side = config.SIDE_CHAR.upper()
me_rois = [f'ME_{side}_layer_{i:02d}' for i in range(1, 11)] + [f'AME({side})']
me_labels = [f'M{i}' for i in range(1, 11)] + ['AME']
lo_rois = (
    [f'LO_{side}_layer_{i}' for i in range(1, 8)]
    + [f'LOP_{side}_layer_{i}' for i in range(1, 5)]
)
lo_labels = ['LO1','LO2','LO3','LO4','LO5A','LO5B','LO6','LOP1','LOP2','LOP3','LOP4']
plot.plot_ol_layers_violin(layers, me_rois, me_labels, filename_stem='ME_layers_hit')
plot.plot_ol_layers_violin(layers, lo_rois, lo_labels, filename_stem='LOPLO_layers_hit')

print('wrote figures to', config.FIG_DIR / 'info_flow')
# SI Fig. 1j: adjacent-anatomical-layer hit-time difference violins
plot.plot_ol_layers_hit_diff(roi_adj, me_rois, me_labels, filename_stem='ME_layers_hit')
plot.plot_ol_layers_hit_diff(roi_adj, lo_rois, lo_labels, filename_stem='LOPLO_layers_hit')


## 3. Clustering and pathways


In [ ]:
FRAC = 0.19

feat        = ol_data.get_ol_features()
clusters    = ol_data.get_ol_clusters(frac_thre=FRAC)
type_part   = ol_data.get_ol_type_participation(frac_thre=FRAC)
intra       = ol_data.get_ol_clusters_intra()
intra_8     = ol_data.get_ol_clusters_intra(frac_thre=0.205)
pathways    = ol_data.get_ol_connectivity_pathways(frac_thre=FRAC)
sankey      = ol_data.get_ol_sankey_connectivity(frac_thre=FRAC)
ol_meta     = core_data.get_ol_meta()
ol_flow     = core_data.get_ol_flow_type()
type_dir_ol = ol_data.get_ol_type_directedness()
sweep       = ol_data.get_ol_lr_sweep()
match       = ol_data.get_ol_lr_cluster_match(frac_thre=FRAC)

n_clu = int(clusters.cluster.max())
cluster_num = clusters['cluster'].value_counts().sort_index()
in_instances = list(ol_meta[ol_meta.main_groups=='visual input'].instance.unique())

colored_region, colored_main_groups, colored_sign, colored_seed = load_colors()

Z, thre, dendr = plot.plot_clusters_dendrograms(feat, frac_thre=FRAC)
plot.plot_clusters_correlation_matrix(feat, dendr, cluster_num)
plot.plot_clusters_example_features(feat, in_instances, [s+'_R' for s in plot._CLUSTER_TYPES], colored_seed)
plot.plot_clusters_pathway_features(feat, in_instances, clusters, cluster_num, colored_seed)
plot.plot_clusters_scatter(clusters, ol_flow, type_dir_ol, colored_main_groups, colored_sign)
plot.plot_clusters_participation(type_part, ol_meta, ol_flow, colored_main_groups, colored_sign)
plot.plot_clusters_intra_connectivity(intra)
plot.plot_clusters_pathway_connectivity(pathways, n_clu)
plot.plot_clusters_sankey(sankey, colored_main_groups, colored_seed, colored_region)
plot.plot_clusters_graph(type_part, ol_meta, colored_main_groups, colored_sign, colored_seed)

# L/R clustering diagnostics (SI Fig. 3d/3f/3j)
plot.plot_lr_clustering_sweep(sweep, n_clu)
plot.plot_lr_cluster_confusion(match)
print('\nOptimal L -> R cluster matching:')
for r, c in zip(match['row_ind'], match['col_ind']):
    print(f'  L cluster {r + 1}  ->  R cluster {c + 1}')
print('wrote figures to', config.FIG_DIR / 'clustering')

# Cluster pathway at FRAC=0.15 (n_clu=13) for SI Fig. 3e filename convention
FRAC_13 = 0.15
clusters_13 = ol_data.get_ol_clusters(frac_thre=FRAC_13)
cluster_num_13 = clusters_13['cluster'].value_counts().sort_index()
plot.plot_clusters_pathway_features(feat, in_instances, clusters_13, cluster_num_13, colored_seed, side_str='_13')

# Left-side OL cluster pathway plots (SI Fig. 3h/3i)
ol_meta_L = core_data.get_ol_meta(side_char='l')
feat_L    = ol_data.get_ol_features(side_char='l')
cl_L      = ol_data.get_ol_clusters(side_char='l', frac_thre=FRAC)
cluster_num_L = cl_L['cluster'].value_counts().sort_index()
in_L = list(ol_meta_L[ol_meta_L.main_groups=='visual input'].instance.unique())
plot.plot_clusters_example_features(feat_L, in_L, [s+'_L' for s in plot._CLUSTER_TYPES], colored_seed, side_str='_left')
plot.plot_clusters_pathway_features(feat_L, in_L, cl_L, cluster_num_L, colored_seed, side_str='_left')
plot.plot_clusters_dendrograms(feat_L, frac_thre=FRAC, side_str='_left')

# SI Fig. 3g outrel_8 at different frac (existing intra is n=13 by default)
plot.plot_clusters_intra_connectivity(intra_8)
print('wrote figures to', config.FIG_DIR / 'pathways')

# Example-neuron pathway diagrams (Main 2c/2f; legacy 0_plot_pathways)
type_to_sign = dict(zip(ol_meta.cell_type, ol_meta.sign))
sign_to_color = colored_sign['color'].to_dict()
type_to_mg = dict(zip(ol_meta.cell_type, colored_main_groups.loc[ol_meta.main_groups, 'color']))
examples_dir = config.FIG_DIR / 'examples'

_layer_cumsum = {'T4a_R': 0.9, 'Mi4_R': 0.7}
for inst in core_data.load_layer_example_types():
    plot.plot_pathway(
        ol_data.get_paths_to_instance(inst), inst, ol_flow,
        thre_cumsum=_layer_cumsum[inst],
        neuron_to_color=type_to_mg,
        neuron_to_sign=type_to_sign, sign_color_map=sign_to_color,
        save_path=examples_dir / f'{config.FIG_DATASET}_{inst[:-2]}_pathway.pdf',
    )

_out_cumsum = {'LC6_R': 0.4, 'LC4_R': 0.6, 'LPLC2_R': 0.6}
for inst, tc in _out_cumsum.items():
    plot.plot_pathway(
        ol_data.get_paths_to_instance(inst), inst, ol_flow,
        thre_cumsum=tc,
        neuron_to_color=type_to_mg,
        neuron_to_sign=type_to_sign, sign_color_map=sign_to_color,
        save_path=examples_dir / f'{config.FIG_DATASET}_{inst[:-2]}_pathway.pdf',
    )

_blues = ['#deebf7','#c6dbef','#9ecae1','#6baed6','#4292c6',
          '#2171b5','#08519c','#08306b','#041f4a']
type_to_color_part = type_to_mg | {
    s.split('_')[0]: _blues[c - 1]
    for s, c in zip(clusters['instance'], clusters['cluster'])
}
for inst in core_data.load_layer_example_types()[:1]:
    plot.plot_pathway(
        ol_data.get_participation_paths(inst), inst, ol_flow,
        thre_cumsum=0.9,
        neuron_to_color=type_to_color_part,
        neuron_to_sign=type_to_sign, sign_color_map=sign_to_color,
        node_size=800, node_text_size=16,
        figsize=(9, 10), frac_text_pos=(0.2, 0.8), frac_text_size=16,
        save_path=examples_dir / f'{config.FIG_DATASET}_{inst[:-2]}_participation_pathway.pdf',
    )
print('wrote pathway examples to', examples_dir)


## 4. OL LR comparison


In [ ]:
FRAC = 0.19

sweep     = ol_data.get_ol_lr_sweep()
match     = ol_data.get_ol_lr_cluster_match(frac_thre=FRAC)
clusters_R= ol_data.get_ol_clusters(frac_thre=FRAC, side_char='r')
n0 = int(clusters_R['cluster'].max())

plot.plot_lr_clustering_sweep(sweep, n0)
plot.plot_lr_cluster_confusion(match)

print('\nOptimal L -> R cluster matching:')
for r, c in zip(match['row_ind'], match['col_ind']):
    print(f'  L cluster {r + 1}  ->  R cluster {c + 1}')
print('wrote figures to', config.FIG_DIR / 'clustering')


## 5. CB clustering and pathways


In [ ]:
FRAC_CB = 0.17
FRAC_OL = 0.19

feat_r    = cb_data.get_cb_features(side_char='r')
cl_r      = cb_data.get_cb_clusters(side_char='r', frac_thre=FRAC_CB)
cluster_num = cl_r['cluster'].value_counts().sort_index()
sweep     = cb_data.get_cb_lr_sweep()
match     = cb_data.get_cb_lr_cluster_match(frac_thre=FRAC_CB)  # threshold-based; left side has 11 clusters
conn      = cb_data.get_ol_cb_cluster_connectivity(ol_frac_thre=FRAC_OL, cb_frac_thre=FRAC_CB)
layer_lr  = cb_data.get_cb_layer_lr()
layer_lr_hom = cb_data.get_cb_layer_lr_homologue()
vic_binoc_type = vic.get_cb_vic_binocular_type()
vic_homologue  = vic.get_cb_vic_lr_homologue()

colored_region, colored_main_groups, _, colored_seed = load_colors()
n0 = int(cl_r['cluster'].max())
in_instances = [c for c in feat_r.columns if c.endswith('_R')]
cb_example_types = plot._CB_CLUSTER_TYPES[config.SIDE_CHAR]

plot.plot_cb_clusters_example_features(feat_r, in_instances, cb_example_types, colored_seed)
plot.plot_cb_clusters_pathway_features(feat_r, in_instances, cl_r, cluster_num, colored_seed)
plot.plot_ol_cb_cluster_connectivity(conn)

plot.plot_cb_lr_clustering_sweep(sweep, n0)
plot.plot_cb_lr_cluster_confusion(match)
plot.plot_cb_layer_lr_comparison(layer_lr, colored_region, colored_main_groups)
plot.plot_cb_layer_lr_homologue(layer_lr_hom, colored_region, colored_main_groups)
plot.plot_cb_vic_binocularity(vic_binoc_type, colored_region)
plot.plot_cb_vic_lr_homologue(vic_homologue, colored_region)

# Left-side CB cluster pathway plots (SI Fig. 4g/4h top + bottom)
# Use the left-side FRAC_CB threshold solution (11 clusters).
feat_cb_L = cb_data.get_cb_features(side_char='l')
cl_cb_L   = cb_data.get_cb_clusters(side_char='l', frac_thre=FRAC_CB)
cluster_num_cb_L = cl_cb_L['cluster'].value_counts().sort_index()
in_cb_L = [c for c in feat_cb_L.columns if c.endswith('_L')]
plot.plot_cb_clusters_example_features(feat_cb_L, in_cb_L, plot._CB_CLUSTER_TYPES['l'], colored_seed, side_str='_left')
plot.plot_cb_clusters_pathway_features(feat_cb_L, in_cb_L, cl_cb_L, cluster_num_cb_L, colored_seed, side_str='_left')
# SI Fig. 4h bottom: left-CB dendrogram
plot.plot_clusters_dendrograms(feat_cb_L, frac_thre=FRAC_CB, side_str='_left_cb', save_dir=config.FIG_DIR / 'CB_pathways', compact_lastp=11)

# CB dendrograms (Main Fig. 4d bot, SI Fig. 4c)
plot.plot_clusters_dendrograms(feat_r, frac_thre=FRAC_CB, side_str='_cb', save_dir=config.FIG_DIR / 'CB_pathways')

# CB cluster-vs-layer scatter (Main Fig. 4e)
plot.plot_cb_clusters_scatter(cl_r, layer_lr, colored_region, highlight=cb_example_types)

# CB intra-cluster connectivity (SI Fig. 4d)
intra_cb = cb_data.get_cb_clusters_intra(frac_thre=FRAC_CB)
plot.plot_cb_clusters_intra_connectivity(intra_cb)

# Main Fig. 4g top: CB cluster × ROI heatmap
tbar_counts = cb_data.get_cb_cluster_tbars_per_roi(side_char='r', cb_frac_thre=FRAC_CB)
plot.plot_cb_clusters_rois(tbar_counts)
# SI Fig. 4k: CB participation n_clusters histogram
type_part_cb_out = cb_data.get_ol_in_cb_type_participation(side_char='r', cb_frac_thre=FRAC_CB)
plot.plot_cb_participation_n_clu_out(type_part_cb_out, colored_main_groups)
# Main Fig. 4f: CB cluster graph
ol_clu = ol_data.get_ol_clusters(side_char='r', frac_thre=FRAC_OL)
_fig4f_types = ["aMe4", "LPT111", "VS", "aMe12", "LC4", "LC6", "LPLC2", "T4a", "Tm5a"]
plot.plot_cb_clusters_graph(type_part_cb_out, ol_clu, colored_region, ol_highlight=_fig4f_types)
# SI Fig. 4a: horizontal left-layer histogram
plot.plot_cb_layer_hist_left(layer_lr_hom, colored_region, colored_main_groups)
# SI Fig. 7c: CB size vs cluster
rf_type_cb_ = ol_rf.get_rf_type_cb()
plot.plot_cb_clusters_size_vs_cluster(cl_r, rf_type_cb_, colored_region)
# SI Fig. 7d: OL->CB high-res subset heatmap
_sizes = ol_rf.get_rf_types_combined().set_index('instance')['size']
plot.plot_out_cb_highres_sub(cb_data.get_ol_to_cb_weights(), _sizes)
print(f'CB clustering @ FRAC_CB={FRAC_CB}: {n0} clusters')
print("left -> right cluster matching:")
for r, c in zip(match['row_ind'], match['col_ind']):
    print(f"  cc'{r + 1}  ->  cc{c + 1}")
print('wrote figures to', config.FIG_DIR / 'CB_pathways', 'and', config.FIG_DIR / 'CB_clustering')

# CB directedness (legacy nb 8 cells 73-74)
dir_type_cb = cb_data.get_cb_type_directedness()
plot.plot_cb_directedness_histograms(dir_type_cb, colored_region)
plot.plot_cb_directedness_triangle(dir_type_cb, colored_region)

# pC1 example-neuron pathway diagrams (Main 4b; legacy 0_plot_pathways CB section)
_meta = core_data.get_meta()
type_to_sign_full = dict(zip(_meta.cell_type, _meta.sign))
sign_to_color_full = colored_sign['color'].to_dict()
# Node colors: main_groups ∪ {nonOL: CB region color} — matches legacy CB section.
_cb_color_df = pd.concat([
    pd.DataFrame(colored_region.loc['CB', 'color'], index=['nonOL'], columns=['color']),
    colored_main_groups,
])
type_to_color_cb = dict(zip(_meta.cell_type, _cb_color_df.loc[_meta.main_groups, 'color']))
flow_type_full = core_data.get_flow().groupby('instance')['hitting_time'].median().reset_index()
examples_dir_cb = config.FIG_DIR / 'examples'

for inst, thre_cumsum, h_scale in [('pC1_2a_R', 0.6, 0.8), ('pC1_5b_R', 0.2, 0.9)]:
    paths = cb_data.get_cb_paths_to_instance(inst)
    hit_inst = paths['hit_inst']
    plot.plot_pathway(
        paths, inst, flow_type_full,
        thre_cumsum=thre_cumsum,
        figsize=(1.5 + hit_inst * 3, 8 * h_scale),
        neuron_to_color=type_to_color_cb,
        neuron_to_sign=type_to_sign_full, sign_color_map=sign_to_color_full,
        node_size=1400, node_text_size=14,
        highlight=False,
        save_path=examples_dir_cb / f'{config.FIG_DATASET}_{inst[:-2]}_pathway.pdf',
    )
print('wrote pC1 pathway examples to', examples_dir_cb)

## 6. Propagated RFs


In [ ]:
FRAC = 0.19

raw       = ol_rf.get_rf_raw_ol()
type_ol   = ol_rf.get_rf_type_ol()
edges_ol  = ol_rf.get_rf_connectivity_edges_ol(include_cb_post=False)
ol_meta   = core_data.get_ol_meta()
stepsn    = ol_data.get_ol_stepsn_sum()
type_part = ol_data.get_ol_type_participation(frac_thre=FRAC)
clusters  = ol_data.get_ol_clusters(frac_thre=FRAC)

colored_region, colored_main_groups, colored_sign, _ = load_colors()
n_clu = int(clusters['cluster'].max())
cluster_names = [f'c{i}' for i in range(1, n_clu + 1)]
in_instances  = ['L1_R', 'L2_R', 'L3_R', 'R7_R', 'R8_R']

EXAMPLE_BID = 30134
plot.plot_rf_example(ol_meta, stepsn, raw, EXAMPLE_BID, in_instances)
plot.plot_rf_positions_across_neurons(raw, 'LC6_R', example_bid=EXAMPLE_BID)

plot_cell_types = [s + '_R' for s in plot._FLOW_PLOT_TYPES]
plot.plot_rf_boxes_by_celltype(raw, plot_cell_types, colored_main_groups)
plot.plot_rf_scatters_by_main_groups(type_ol, colored_main_groups, colored_sign)
plot.plot_rf_scatters_by_cluster(
    type_ol, type_part, cluster_names=cluster_names,
    colored_main_groups_df=colored_main_groups,
)
plot.plot_rf_histograms(
    type_ol, type_part, cluster_names=cluster_names,
    colored_main_groups_df=colored_main_groups,
    colored_sign_df=colored_sign,
)
plot.plot_rf_sizediff_internal(edges_ol, colored_main_groups)
plot.plot_rf_connectivity_boxes(edges_ol, colored_main_groups, colored_sign)

print('wrote figures to', config.FIG_DIR / 'rf_fits')
# Main Fig. 3a pos_par overlay
plot.plot_rf_example_pos_par(ol_meta, stepsn, raw, EXAMPLE_BID, in_instances)

# Main Fig. 3c: direct-input RFs for LC6_R (legacy find_direct_rfs cells 15/18/22)
roi_counts = pd.read_pickle(
    config.DATA_DIR / f'{config.DATASET}_OL_{config.SIDE_CHAR}_roi_counts.pkl'
)
input_raw = ol_rf.get_input_rf_raw_ol()
plot.plot_input_rf_example(roi_counts, input_raw, EXAMPLE_BID, 'LC6_R')


## 7. RF size experimental comparisons


In [ ]:
cmp_body = ol_rf.get_rf_comparison_body()
cmp_type = ol_rf.get_rf_comparison_type()
exp_sz   = ol_rf.get_experimental_rf_sizes()
exp_cc   = ol_rf.get_experimental_rf_sizes(sheet='RF size CC')
type_ol  = ol_rf.get_rf_type_ol()
colored_region, colored_main_groups, colored_sign, _ = load_colors()

EXAMPLE_BID = 30134

plot.plot_rf_size_example_comparison(cmp_body, star_instance='LC6_R', example_bid=EXAMPLE_BID)
plot.plot_rf_size_type_scatters(cmp_type, colored_main_groups)
plot.plot_rf_size_vs_experiment(cmp_type, exp_sz, colored_main_groups, colored_region)
plot.plot_rf_size_vs_experiment_cc(type_ol, exp_cc, colored_main_groups, colored_region)

print('wrote figures to', config.FIG_DIR / 'exp_comparison')


## 8. OL polarity experimental comparisons


In [ ]:
cmp_ol = polarity.get_ol_polarity_comparison()
colored_region, colored_main_groups, colored_sign, colored_seed = load_colors()

cmp_vpn   = cmp_ol[cmp_ol['main_groups'] == 'OL output'].reset_index(drop=True)
cmp_other = cmp_ol[cmp_ol['main_groups'] != 'OL output'].reset_index(drop=True)

plot.plot_polarity_stacked_bars(cmp_vpn, colored_seed, filename_stem='polarity_predictions_vpn')
plot.plot_polarity_stacked_bars(
    cmp_other, colored_seed, filename_stem='polarity_predictions_other',
    width=int(max(400, 35 * len(cmp_other) + 150) * 0.704),
)

print(f'  VPN (OL output): {cmp_vpn.match.mean():.1%} ({len(cmp_vpn)} types)')
print(f'  Other OL:        {cmp_other.match.mean():.1%} ({len(cmp_other)} types)')
miss = cmp_ol[~cmp_ol.match][['instance', 'polarity', 'predicted', 'main_groups']]
print(f'  OL mismatches ({len(miss)}):')
print(miss.to_string(index=False))
print('wrote figures to', config.FIG_DIR / 'exp_comparison')

ol_meta = core_data.get_ol_meta()
ol_flow = core_data.get_ol_flow_type()
type_to_sign = dict(zip(ol_meta.cell_type, ol_meta.sign))
sign_to_color = colored_sign['color'].to_dict()
type_to_mg = dict(zip(ol_meta.cell_type, colored_main_groups.loc[ol_meta.main_groups, 'color']))
examples_dir = config.FIG_DIR / 'examples'

_ol_miss_cumsum_override = {'DN1a_R': 0.6, 'Tm29_R': 0.6}
for inst, mg in zip(miss.instance, miss.main_groups):
    tc = _ol_miss_cumsum_override.get(inst, 0.4 if mg == 'OL output' else 0.8)
    plot.plot_pathway(
        ol_data.get_paths_to_instance(inst), inst, ol_flow,
        thre_cumsum=tc,
        neuron_to_color=type_to_mg,
        neuron_to_sign=type_to_sign, sign_color_map=sign_to_color,
        save_path=examples_dir / f'{config.FIG_DATASET}_{inst[:-2]}_pathway.pdf',
    )
print(f'wrote {len(miss)} OL mismatch pathways to', examples_dir)


## 9. CB VICs + CB RFs


In [ ]:
vic_type = vic.get_ol_cb_vic_type()
flow     = core_data.get_flow()
vic_type = vic_type.merge(
    flow.groupby('instance')['hitting_time'].median().reset_index(),
    on='instance', how='left',
)
rf_all   = ol_rf.get_rf_types_combined()
rf_vpn_vcbn = rf_all[(rf_all['main_groups'] == 'OL output') | (rf_all['region'] == 'CB')]
edges_cb = ol_rf.get_rf_connectivity_edges_ol(include_cb_post=True)
tb_clu   = cb_data.get_ol_to_cb_weights()
colored_region, colored_main_groups, colored_sign, _ = load_colors()

# VPNs colored by OL output blue (main_groups), CB stays dark gray (region) — mirrors Main 4a / SI 7a-b.
_vpn_colors = colored_region.copy()
_vpn_colors.loc['right OL', 'color'] = colored_main_groups.loc['OL output', 'color']

plot.plot_vic_cumsum(vic_type, _vpn_colors)
plot.plot_vic_layer_histogram(vic_type, _vpn_colors)
plot.plot_vic_vs_layer_scatter(vic_type, _vpn_colors)

plot.plot_rf_sizediff_output(edges_cb, colored_region)

plot.plot_cb_rf_r2_summary(rf_vpn_vcbn, _vpn_colors)
plot.plot_cb_rf_size_summary(rf_vpn_vcbn, _vpn_colors)
plot.plot_ol_to_cb_heatmap(tb_clu)

print('wrote figures to', config.FIG_DIR / 'CB_pathways', ',', config.FIG_DIR / 'rf_fits', 'and', config.FIG_DIR / 'CB_rf_fits')


## 10. CB polarity experimental comparisons


In [ ]:
import pandas as pd

In [ ]:
cmp_cb = polarity.get_cb_polarity_comparison()
colored_region, colored_main_groups, colored_sign, colored_seed = load_colors()

plot.plot_polarity_stacked_bars(cmp_cb, colored_seed, filename_stem='polarity_predictions_cb',
    width=int(max(400, 35 * len(cmp_cb) + 150) * 0.96),
    font_size=14)

print(f'  CB: {cmp_cb.match.mean():.1%} ({len(cmp_cb)} types)')
miss_cb = cmp_cb[~cmp_cb.match][['instance', 'polarity', 'predicted']]
print(f'  CB mismatches ({len(miss_cb)}):')
print(miss_cb.to_string(index=False))
print('wrote figures to', config.FIG_DIR / 'exp_comparison')

_cl_r = cb_data.get_cb_clusters(side_char='r', frac_thre=0.17)

def _expand_range_instance(inst):
    """Expand range-notation instance names:
    - 'TuBu02-05_R'    -> ['TuBu02_R', ..., 'TuBu05_R']  (numeric, letter-only prefix)
    - 'DNg02_a-f_R'    -> ['DNg02_a_R', ..., 'DNg02_f_R'] (alphabetic single-char suffix)
    - 'PVLP008_a1-4_R' -> ['PVLP008_a1_R', ..., 'PVLP008_a4_R'] (numeric, any prefix)
    """
    import re
    # Numeric with letter-only prefix: TuBu02-05_R
    m = re.fullmatch(r'([A-Za-z_]+)(\d+)-(\d+)(_[LR])', inst)
    if m:
        prefix, start, end, suffix = m.group(1), int(m.group(2)), int(m.group(3)), m.group(4)
        width = len(m.group(2))
        return [f'{prefix}{str(i).zfill(width)}{suffix}' for i in range(start, end + 1)]
    # Alphabetic single-char suffix: DNg02_a-f_R
    m = re.fullmatch(r'(.+)_([a-z])-([a-z])(_[LR])', inst)
    if m:
        prefix, start_c, end_c, suffix = m.group(1), m.group(2), m.group(3), m.group(4)
        return [f'{prefix}_{chr(c)}{suffix}' for c in range(ord(start_c), ord(end_c) + 1)]
    # Numeric with any prefix: PVLP008_a1-4_R
    m = re.fullmatch(r'(.+?)(\d+)-(\d+)(_[LR])', inst)
    if m:
        prefix, start, end, suffix = m.group(1), int(m.group(2)), int(m.group(3)), m.group(4)
        width = len(m.group(2))
        return [f'{prefix}{str(i).zfill(width)}{suffix}' for i in range(start, end + 1)]
    return [inst]

_cmp_rows = []
for _, _row in cmp_cb[['instance', 'polarity', 'predicted', 'match']].iterrows():
    for _inst in _expand_range_instance(_row['instance']):
        _cmp_rows.append({**_row.to_dict(), 'instance': _inst})
_cmp_clu = pd.DataFrame(_cmp_rows).merge(
    _cl_r[['instance', 'cluster']], on='instance', how='left'
)
print(f'\nCluster assignments for polarity comparison types:')
print(_cmp_clu.to_string(index=False))

_meta = core_data.get_meta()
type_to_sign_full = dict(zip(_meta.cell_type, _meta.sign))
sign_to_color_full = colored_sign['color'].to_dict()
_cb_color_df = pd.concat([
    pd.DataFrame(colored_region.loc['CB', 'color'], index=['nonOL'], columns=['color']),
    colored_main_groups,
])
type_to_color_cb = dict(zip(_meta.cell_type, _cb_color_df.loc[_meta.main_groups, 'color']))
flow_type_full = core_data.get_flow().groupby('instance')['hitting_time'].median().reset_index()
examples_dir_cb = config.FIG_DIR / 'examples'

for inst in miss_cb.instance:
    plot.plot_pathway(
        cb_data.get_cb_paths_to_instance(inst), inst, flow_type_full,
        thre_cumsum=0.4,
        neuron_to_color=type_to_color_cb,
        neuron_to_sign=type_to_sign_full, sign_color_map=sign_to_color_full,
        node_size=1400, node_text_size=14,
        highlight=False,
        save_path=examples_dir_cb / f'{config.FIG_DATASET}_{inst[:-2]}_pathway.pdf',
    )
print(f'wrote {len(miss_cb)} CB mismatch pathways to', examples_dir_cb)

## 11. CB RFs


In [ ]:
coverage = cb_data.get_ol_roi_coverage(seed=0)
sectors  = core_data.get_sector_map()
plot.plot_ol_roi_coverage_per_roi(coverage, sectors)
plot.plot_ol_roi_coverage_summary(coverage)

rf_all = ol_rf.get_rf_types_combined()
plot.plot_rf_size_per_layer(rf_all)
print('wrote figures to', config.FIG_DIR / 'coverage', 'and', config.FIG_DIR / 'CB_rf_fits')

## 12. Sankey diagram for full visual system


In [ ]:
FRAC_OL = 0.19
FRAC_CB = 0.17

visual_sankey = cb_data.get_visual_system_sankey_connectivity(
    ol_frac_thre=FRAC_OL,
    cb_frac_thre=FRAC_CB,
    min_plot_neurons=10,
)
colored_region, colored_main_groups, _, colored_seed = load_colors()
plot.plot_visual_system_sankey(
    visual_sankey,
    colored_main_groups,
    colored_seed,
    colored_region,
)
print('wrote figures to', config.FIG_DIR / 'CB_pathways')